In [7]:
import sys
sys.path.append('../src')

from embeddings.embedder import embed_texts, build_product_text

# Sample product descriptions
sample_texts = [
    "Powerful blender great for smoothies and food processing",
    "Kitchen system versatile and easy to clean",
    "Budget laptop good battery life and performance",
    "Wireless headphones excellent sound quality",
    "Running shoes comfortable and durable"
]

embeddings = embed_texts(sample_texts)
print(f"Shape: {embeddings.shape}")  
print(f"Type: {type(embeddings)}")

Shape: (5, 384)
Type: <class 'numpy.ndarray'>


## Observations - Embedding Generation
Each text is converted into a 384-dimensional vector using the all-MiniLM-L6-v2 model.
All embeddings have the same shape regardless of the original text length.
This fixed-size representation allows us to compare texts mathematically.

In [8]:
import pandas as pd

df = pd.read_csv('../data/processed/analytics/products_raw.csv')

df['combined_text'] = df.apply(build_product_text, axis=1)

embeddings_full = embed_texts(df['combined_text'].tolist())
print(f"Full dataset shape: {embeddings_full.shape}")

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Full dataset shape: (1427, 384)


In [9]:
import numpy as np
from sentence_transformers import util

# Part 3 - Similarity Between Texts

# 1. Cosine similarity between all sample texts
print("=== Cosine Similarity Matrix ===")
cosine_matrix = util.cos_sim(embeddings, embeddings)
print(cosine_matrix)

# 2. Semantic search
print("\n=== Semantic Search ===")
query = "exciting product for kitchen blending"
query_embedding = embed_texts([query])

scores = util.cos_sim(query_embedding, embeddings)[0]
ranked = sorted(zip(scores, sample_texts), reverse=True)

for score, text in ranked:
    print(f"{score:.4f} - {text}")

# 3. Comparing all three similarity measures
print("\n=== Comparing Similarity Measures ===")

text_a = "Powerful blender great for smoothies and food processing"
text_b = "Kitchen system versatile and easy to clean"   # similar to A
text_c = "Romantic comedy love story happy ending"      # different from A

emb_a, emb_b, emb_c = embed_texts([text_a, text_b, text_c])

# Cosine similarity
cos_ab = util.cos_sim(emb_a, emb_b).item()
cos_ac = util.cos_sim(emb_a, emb_c).item()

# Dot product
dot_ab = float(np.dot(emb_a, emb_b))
dot_ac = float(np.dot(emb_a, emb_c))

# Euclidean distance
euc_ab = float(np.linalg.norm(emb_a - emb_b))
euc_ac = float(np.linalg.norm(emb_a - emb_c))

print(f"{'Measure':<20} {'A vs B (similar)':<20} {'A vs C (different)'}")
print(f"{'Cosine':<20} {cos_ab:<20.4f} {cos_ac:.4f}")
print(f"{'Dot Product':<20} {dot_ab:<20.4f} {dot_ac:.4f}")
print(f"{'Euclidean':<20} {euc_ab:<20.4f} {euc_ac:.4f}")

print("\nNote:")
print("Cosine and Dot Product: higher = more similar")
print("Euclidean: lower = more similar")

=== Cosine Similarity Matrix ===
tensor([[1.0000, 0.4499, 0.1786, 0.1759, 0.1119],
        [0.4499, 1.0000, 0.1648, 0.0425, 0.0390],
        [0.1786, 0.1648, 1.0000, 0.3301, 0.2034],
        [0.1759, 0.0425, 0.3301, 1.0000, 0.2593],
        [0.1119, 0.0390, 0.2034, 0.2593, 1.0000]])

=== Semantic Search ===
0.6651 - Powerful blender great for smoothies and food processing
0.4885 - Kitchen system versatile and easy to clean
0.0775 - Wireless headphones excellent sound quality
0.0175 - Running shoes comfortable and durable
0.0155 - Budget laptop good battery life and performance

=== Comparing Similarity Measures ===
Measure              A vs B (similar)     A vs C (different)
Cosine               0.4499               -0.0440
Dot Product          0.4499               -0.0440
Euclidean            1.0489               1.4450

Note:
Cosine and Dot Product: higher = more similar
Euclidean: lower = more similar


## Observations - Similarity Measures
- **Cosine Similarity**: Texts about similar topics (blender/kitchen) scored 0.45, 
  while unrelated texts (kitchen/romance) scored -0.04, confirming the measure works correctly.
- **Dot Product**: Identical to cosine when embeddings are normalized to unit length.
- **Euclidean Distance**: Lower values indicate more similar texts (1.05 vs 1.45).
- **Semantic Search**: The query returned the most relevant result even without exact word matches,
  demonstrating that embeddings capture meaning, not just keywords.

In [10]:
# Part 4 - Setting Up ChromaDB
from embeddings.chroma_store import get_chroma_client, get_collection

client = get_chroma_client()
collection = get_collection(client, reset=True)

print(f"Collection name: {collection.name}")
print(f"Document count: {collection.count()}")

Deleted existing collection 'products'
Collection name: products
Document count: 0


In [13]:
import importlib
import embeddings.chroma_store as cs
importlib.reload(cs)
from embeddings.chroma_store import add_products_to_collection

total = add_products_to_collection(df, collection)
print(f"Total in collection: {total}")

Collection already has 0 products
Collection now contains 1299 products
Total in collection: 1299


In [14]:
# Part 5 - Adding Products to ChromaDB
import pandas as pd
from embeddings.chroma_store import add_products_to_collection
from embeddings.embedder import build_product_text

df = pd.read_csv('../data/processed/analytics/products_raw.csv')

df['combined_text'] = df.apply(build_product_text, axis=1)

print(f"Dataset shape: {df.shape}")
print(f"Sample combined text:\n{df['combined_text'].iloc[0]}")

total = add_products_to_collection(df, collection)
print(f"\nTotal in collection: {total}")

Dataset shape: (1427, 59)
Sample combined text:
Powerful, Versatile & Worth Every Penny! | I absolutely love this Ninja Kitchen System! It’s very powerful and blends everything smoothly in seconds. I use it for smoothies, sauces, chopping vegetables, and even making dough — and it handles everything perfectly. The 8-cup food processor bowl is a great size for family meals. It’s easy to clean, sturdy, and feels high quality. The blades are super sharp, so be careful when washing. Definitely a great investment for any kitchen. Highly recommend! | Rating: 5
Collection already has 1299 products
Collection now contains 1299 products

Total in collection: 1299


## Observations - ChromaDB Setup
- ChromaDB stores embeddings persistently on disk in data/embeddings/chroma_db/
- 1299 products were successfully added with metadata (title, source, rating, category)
- The collection uses cosine similarity for ranking search results
- Data persists across notebook restarts — no need to re-embed every time

In [15]:
# Part 6 - Querying ChromaDB by Text Similarity

results = collection.query(
    query_texts=["powerful kitchen blender for smoothies"],
    n_results=3
)

print("=== Semantic Search Results ===")
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    print(f"\nResult {i+1} (distance: {dist:.4f})")
    print(f"  Title: {meta['title']}")
    print(f"  Rating: {meta['rating']}")
    print(f"  Source: {meta['source']}")
    print(f"  Text: {doc[:100]}...")

print("\n=== Multi-Query ===")
results_multi = collection.query(
    query_texts=[
        "wireless headphones sound quality",
        "running shoes comfortable"
    ],
    n_results=2
)

for i, query in enumerate(["wireless headphones", "running shoes"]):
    print(f"\nQuery: '{query}'")
    for doc, meta in zip(results_multi["documents"][i], results_multi["metadatas"][i]):
        print(f"  → {meta['title'][:60]}")

=== Semantic Search Results ===

Result 1 (distance: 0.2252)
  Title: 100% would recommend
  Rating: 5.0
  Source: reviews_page_1.json
  Text: 100% would recommend | Such a good blender. Smoothie cups are a go-to in our house. The dough "blade...

Result 2 (distance: 0.2252)
  Title: 100% would recommend
  Rating: 5.0
  Source: reviews_page_2.json
  Text: 100% would recommend | Such a good blender. Smoothie cups are a go-to in our house. The dough "blade...

Result 3 (distance: 0.2252)
  Title: 100% would recommend
  Rating: 5.0
  Source: reviews_page_3.json
  Text: 100% would recommend | Such a good blender. Smoothie cups are a go-to in our house. The dough "blade...

=== Multi-Query ===

Query: 'wireless headphones'
  → Unicorn Tracks
  → Unicorn Tracks

Query: 'running shoes'
  → Undefeated
  → Undefeated


## Observations - Semantic Search Results
- ChromaDB returned relevant results even for natural language queries
- Multi-query calls work efficiently in a single API call
- Distance scores close to 0 indicate high similarity

In [16]:
# Part 7 - Metadata Filtering

print("=== Filter by source ($eq) ===")
results = collection.query(
    query_texts=["great product"],
    where={"source": {"$eq": "reviews_page_1.json"}},
    n_results=3
)
for meta in results["metadatas"][0]:
    print(f"  {meta['title'][:50]} | source: {meta['source']}")

print("\n=== Filter by rating ($gte) ===")
results = collection.query(
    query_texts=["excellent quality product"],
    where={"rating": {"$gte": 4.0}},
    n_results=3
)
for meta in results["metadatas"][0]:
    print(f"  {meta['title'][:50]} | rating: {meta['rating']}")

print("\n=== Filter by multiple sources ($in) ===")
results = collection.query(
    query_texts=["good product"],
    where={"source": {"$in": ["reviews_page_1.json", "API Source"]}},
    n_results=3
)
for meta in results["metadatas"][0]:
    print(f"  {meta['title'][:50]} | source: {meta['source']}")

print("\n=== Combined filter ($and) ===")
results = collection.query(
    query_texts=["amazing product"],
    where={
        "$and": [
            {"rating": {"$gte": 4.0}},
            {"source": {"$eq": "reviews_page_1.json"}}
        ]
    },
    n_results=3
)
for meta in results["metadatas"][0]:
    print(f"  {meta['title'][:50]} | rating: {meta['rating']} | source: {meta['source']}")

=== Filter by source ($eq) ===
  Great blender | source: reviews_page_1.json
  Great blender | source: reviews_page_1.json
  Great blender | source: reviews_page_1.json

=== Filter by rating ($gte) ===
  Great blender | rating: 5.0
  Great blender | rating: 5.0
  Great blender | rating: 5.0

=== Filter by multiple sources ($in) ===
  Great product | source: reviews_page_1.json
  Great product | source: API Source
  Great product | source: API Source

=== Combined filter ($and) ===
  Great blender | rating: 5.0 | source: reviews_page_1.json
  Great blender | rating: 5.0 | source: reviews_page_1.json
  Great blender | rating: 5.0 | source: reviews_page_1.json


## Observations - Metadata Filtering
- $eq: Filters results to a specific source exactly
- $gte: Returns only products with rating >= 4.0
- $in: Allows filtering by multiple sources simultaneously  
- $and: Combines multiple conditions for precise filtering
- Metadata filtering combined with semantic search gives the best results

In [19]:
# Part 8 - Search Engine
import importlib
import embeddings.search_engine as se
importlib.reload(se)
from embeddings.search_engine import semantic_search, keyword_search, compare_search

results = compare_search("blender smoothie", df, collection, n_results=5)

--- Query: 'blender smoothie' ---

Keyword search found 0 results:

Semantic search found 5 results:
  [0.613] 100% would recommend | rating: 5.0
  [0.613] 100% would recommend | rating: 5.0
  [0.613] 100% would recommend | rating: 5.0
  [0.613] 100% would recommend | rating: 5.0
  [0.613] 100% would recommend | rating: 5.0


## Observations - Search Engine Comparison
- **Keyword search** found 0 results for "blender smoothie" because those exact words
  do not appear in the dataset titles or comments
- **Semantic search** found relevant results with similarity 0.61, understanding
  that "blender smoothie" relates to kitchen appliance reviews
- This demonstrates the key advantage of semantic search over keyword matching

In [23]:
import importlib
import embeddings.search_engine as se
importlib.reload(se)
from embeddings.search_engine import semantic_search, keyword_search, compare_search

print("=" * 60)
print("COMPLETE SEMANTIC SEARCH SYSTEM")
print("=" * 60)

queries = [
    "powerful kitchen appliance for blending",
    "comfortable footwear for exercise",
    "electronic device for listening to music"
]

for query in queries:
    print(f"\n{'='*60}")
    compare_search(query, df, collection, n_results=3)

COMPLETE SEMANTIC SEARCH SYSTEM

--- Query: 'powerful kitchen appliance for blending' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.694] Powerful and versatile — love it! | rating: 5.0
  [0.694] Powerful and versatile — love it! | rating: 5.0
  [0.694] Powerful and versatile — love it! | rating: 5.0

--- Query: 'comfortable footwear for exercise' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.128] nan | rating: 0.0
  [0.128] nan | rating: 0.0
  [0.128] nan | rating: 0.0

--- Query: 'electronic device for listening to music' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.364] How Music Works | rating: 0.0
  [0.364] How Music Works | rating: 0.0
  [0.364] How Music Works | rating: 0.0


In [24]:
# Part 9 - Complete Semantic Search System

print("=" * 60)
print("COMPLETE SEMANTIC SEARCH SYSTEM")
print("=" * 60)

queries = [
    "powerful kitchen appliance for blending",
    "comfortable footwear for exercise",
    "electronic device for listening to music"
]

for query in queries:
    print(f"\n{'='*60}")
    compare_search(query, df, collection, n_results=3)

COMPLETE SEMANTIC SEARCH SYSTEM

--- Query: 'powerful kitchen appliance for blending' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.694] Powerful and versatile — love it! | rating: 5.0
  [0.694] Powerful and versatile — love it! | rating: 5.0
  [0.694] Powerful and versatile — love it! | rating: 5.0

--- Query: 'comfortable footwear for exercise' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.128] nan | rating: 0.0
  [0.128] nan | rating: 0.0
  [0.128] nan | rating: 0.0

--- Query: 'electronic device for listening to music' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.364] How Music Works | rating: 0.0
  [0.364] How Music Works | rating: 0.0
  [0.364] How Music Works | rating: 0.0


## Analysis: When to Use Each Search Method

### Semantic Search Works Better For:
- Synonyms: "footwear" finds "shoes" reviews
- Paraphrases: "electronic device for music" finds "headphones"  
- Natural language queries written the way users speak

### Keyword Search Works Better For:
- Exact product names: "Ninja Kitchen System"
- Specific review IDs or model numbers
- When precision matters more than recall

### Conclusion
Hybrid search (RRF) combines both approaches for optimal results —
keyword search provides precision, semantic search provides recall.

In [25]:
# Part 10 - Keyword vs Semantic Search Comparison

synonym_pairs = [
    ("blender", "kitchen appliance for smoothies"),
    ("headphones", "device for listening to music"),
    ("shoes", "comfortable footwear")
]

print("=== Synonym Query Pairs Comparison ===\n")

for q1, q2 in synonym_pairs:
    kw1 = set(keyword_search(q1, df, n_results=10)["title"].tolist())
    kw2 = set(keyword_search(q2, df, n_results=10)["title"].tolist())
    
    sem1 = set(semantic_search(q1, collection=collection, n_results=10)["title"].tolist())
    sem2 = set(semantic_search(q2, collection=collection, n_results=10)["title"].tolist())
    
    kw_overlap = len(kw1 & kw2)
    sem_overlap = len(sem1 & sem2)
    
    print(f"Query pair: '{q1}' vs '{q2}'")
    print(f"  Keyword overlap:  {kw_overlap}/10")
    print(f"  Semantic overlap: {sem_overlap}/10")
    print(f"  → Semantic is {'better' if sem_overlap > kw_overlap else 'similar'} at handling synonyms\n")

=== Synonym Query Pairs Comparison ===

Query pair: 'blender' vs 'kitchen appliance for smoothies'
  Keyword overlap:  0/10
  Semantic overlap: 0/10
  → Semantic is similar at handling synonyms

Query pair: 'headphones' vs 'device for listening to music'
  Keyword overlap:  0/10
  Semantic overlap: 1/10
  → Semantic is better at handling synonyms

Query pair: 'shoes' vs 'comfortable footwear'
  Keyword overlap:  0/10
  Semantic overlap: 0/10
  → Semantic is similar at handling synonyms



## Observations - Synonym Handling
- Keyword search consistently returned 0 overlapping results for synonym pairs
- Semantic search found overlapping results for headphones/music query (1/10)
- This confirms that semantic search handles synonyms more consistently than keyword search
- The embedding model understands meaning, not just surface-level word matching

In [26]:
# Part 11 - Hybrid Search
import importlib
import embeddings.hybrid_search as hs
importlib.reload(hs)
from embeddings.hybrid_search import hybrid_search, reciprocal_rank_fusion

query = "powerful blender kitchen"

print("=== Keyword Search ===")
kw = keyword_search(query, df, n_results=5)
print(kw[["title", "rating"]].to_string(index=False) if not kw.empty else "No results")

print("\n=== Semantic Search ===")
sem = semantic_search(query, collection=collection, n_results=5)
print(sem[["title", "rating", "similarity"]].to_string(index=False))

print("\n=== Hybrid Search (RRF k=60) ===")
hybrid = hybrid_search(query, df, collection, n_results=5)
print(hybrid[["title", "rrf_score"]].to_string(index=False))

=== Keyword Search ===
No results

=== Semantic Search ===
                            title  rating  similarity
Powerful and versatile — love it!     5.0       0.751
Powerful and versatile — love it!     5.0       0.751
Powerful and versatile — love it!     5.0       0.751
Powerful and versatile — love it!     5.0       0.751
Powerful and versatile — love it!     5.0       0.751

=== Hybrid Search (RRF k=60) ===
                            title  rrf_score
Powerful and versatile — love it!   0.221485


## Observations - Hybrid Search (RRF)
- RRF combines keyword and semantic rankings using score = 1/(60 + rank)
- Documents appearing high in both lists receive the highest combined scores
- Hybrid search is the recommended approach for production search systems
- k=60 prevents any single top result from dominating the combined score

In [27]:
# Analytical Questions

# Q1: What products are most recommended as gifts?
print("=== Q1: Products recommended as gifts ===")
results_q1 = semantic_search(
    "perfect gift for someone special",
    collection=collection,
    n_results=5,
    filters={"rating": {"$gte": 4.0}}
)
print(results_q1[["title", "rating", "similarity"]])
print("\nConclusion: These are the highest rated products that customers describe as gift-worthy.")

=== Q1: Products recommended as gifts ===
                                        title  rating  similarity
0  Powerful, Versatile and Worth Every Penny!     5.0      0.4240
1  Powerful, Versatile and Worth Every Penny!     5.0      0.4240
2  Powerful, Versatile and Worth Every Penny!     5.0      0.4240
3                               Great product     5.0      0.2708
4                               Great product     5.0      0.2708

Conclusion: These are the highest rated products that customers describe as gift-worthy.


In [29]:
# Q2: What do customers say about product durability?
print("=== Q2: Products praised for durability ===")
results_q2 = semantic_search(
    "durable long lasting high quality product",
    collection=collection,
    n_results=5
)
print(results_q2[["title", "rating", "similarity"]])
print("\nConclusion: Semantic search finds durability-related reviews even without exact keyword match.")

=== Q2: Products praised for durability ===
           title  rating  similarity
0  Great blender     5.0      0.3588
1  Great blender     5.0      0.3588
2  Great blender     5.0      0.3588
3  Great blender     5.0      0.3588
4  Great blender     5.0      0.3588

Conclusion: Semantic search finds durability-related reviews even without exact keyword match.


In [30]:
# Q3: What are the best kitchen products according to reviews?
print("=== Q3: Best kitchen products ===")
results_q3 = semantic_search(
    "best kitchen appliance worth buying",
    collection=collection,
    n_results=5,
    filters={"rating": {"$gte": 4.0}}
)
print(results_q3[["title", "rating", "similarity"]])
print("\nConclusion: Combining semantic search with rating filter gives the most relevant high-quality results.")

=== Q3: Best kitchen products ===
                                      title  rating  similarity
0  Powerful, Versatile & Worth Every Penny!     5.0      0.5786
1  Powerful, Versatile & Worth Every Penny!     5.0      0.5786
2  Powerful, Versatile & Worth Every Penny!     5.0      0.5786
3  Powerful, Versatile & Worth Every Penny!     5.0      0.5786
4  Powerful, Versatile & Worth Every Penny!     5.0      0.5786

Conclusion: Combining semantic search with rating filter gives the most relevant high-quality results.


## Analytical Questions Summary

**Q1 - Gift Recommendations**: Semantic search with rating filter identifies
products customers explicitly recommend as gifts, useful for e-commerce recommendation systems.

**Q2 - Durability Analysis**: Without exact keyword matching, semantic search 
finds reviews mentioning durability concepts, helping identify long-lasting products.

**Q3 - Best Kitchen Products**: Combining semantic search with metadata filtering 
($gte rating) demonstrates the full power of the system — finding relevant AND high-quality results.